In [ ]:
from pathlib import Path
import cv2
from tqdm import tqdm
import pandas as pd
import numpy as np

In [21]:
def mask_to_bbox(mask):
    """Convert mask to bounding box with validation"""
    if mask is None:
        return None
    
    # Ensure binary mask
    if mask.dtype != np.uint8:
        mask = (mask > 0).astype(np.uint8) * 255
    
    # Find contours for more accurate bounding box
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        # Fallback to direct coordinate extraction
        ys, xs = np.where(mask > 0)
        if len(xs) == 0 or len(ys) == 0:
            return None
        x1, y1 = xs.min(), ys.min()
        x2, y2 = xs.max(), ys.max()
    else:
        # Use the largest contour (main tumor)
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        x1, y1, x2, y2 = x, y, x + w, y + h
    
    # Validate bounding box
    if x1 >= x2 or y1 >= y2:
        return None
    
    return [int(x1), int(y1), int(x2), int(y2)]

In [22]:
def crop_with_margin(image, box, margin=0.15):
    """Crop image with margin around bounding box"""
    x1, y1, x2, y2 = box
    h, w = image.shape[:2]

    bw = x2 - x1
    bh = y2 - y1

    pad_x = int(bw * margin)
    pad_y = int(bh * margin)

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x)
    y2 = min(h, y2 + pad_y)

    return image[y1:y2, x1:x2]


# ============================================
# CROP IMAGES AND MASKS USING BOUNDING BOXES
# ============================================

# Set paths
dataset_root = Path("../../dataset/BUSI_Jpeg")
crop_root = Path("../../dataset/cropped_tumor_images")

# Create output directories
crop_root.mkdir(parents=True, exist_ok=True)

saved_crop_rows = []

# Process each class (benign and malignant only)
for class_name in ['benign', 'malignant']:
    img_dir = dataset_root / class_name
    mask_dir = dataset_root / f"{class_name}_mask"
    
    if not img_dir.exists() or not mask_dir.exists():
        print(f"Skipping {class_name}: directories not found")
        continue
    
    # Get all image files
    img_files = list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg"))
    img_files = [f for f in img_files if not f.name.startswith('._')]
    
    print(f"\nProcessing {class_name}: {len(img_files)} images")
    
    for img_path in tqdm(img_files, desc=f"Cropping {class_name}"):
        # Find corresponding mask
        stem = img_path.stem
        mask_path = mask_dir / f"{stem}_mask.png"
        
        if not mask_path.exists():
            continue
        
        # Load images
        img = cv2.imread(str(img_path))
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        
        if img is None or mask is None:
            continue
        
        # Get bounding box from mask
        box = mask_to_bbox(mask)
        if box is None:
            continue
        
        # Crop both image and mask with same bounding box
        img_crop = crop_with_margin(img, box, margin=0.15)
        mask_crop = crop_with_margin(mask, box, margin=0.15)
        
        if img_crop is None or img_crop.size == 0 or mask_crop is None or mask_crop.size == 0:
            continue
        
        # Save cropped image and mask
        save_dir = crop_root / class_name
        img_save_dir = save_dir / "images"
        mask_save_dir = save_dir / "masks"
        
        img_save_dir.mkdir(parents=True, exist_ok=True)
        mask_save_dir.mkdir(parents=True, exist_ok=True)
        
        img_crop_path = img_save_dir / f"{stem}.png"
        mask_crop_path = mask_save_dir / f"{stem}_mask.png"
        
        cv2.imwrite(str(img_crop_path), img_crop)
        cv2.imwrite(str(mask_crop_path), mask_crop)
        
        saved_crop_rows.append({
            "image_path": str(img_path),
            "mask_path": str(mask_path),
            "class_name": class_name,
            "crop_image_path": str(img_crop_path),
            "crop_mask_path": str(mask_crop_path),
            "bbox_xyxy": box
        })

# Create DataFrame
cropped_df = pd.DataFrame(saved_crop_rows)

print("\n" + "="*50)
print("CROPPING COMPLETE")
print("="*50)
print(f"Total cropped images: {len(cropped_df)}")
print(f"Class distribution:")
print(cropped_df["class_name"].value_counts())
print(f"\nOutput directory structure:")
print(f"  {crop_root}/benign/images/     - Cropped images")
print(f"  {crop_root}/benign/masks/      - Cropped masks")
print(f"  {crop_root}/malignant/images/  - Cropped images")
print(f"  {crop_root}/malignant/masks/   - Cropped masks")
cropped_df.head()


Processing benign: 437 images


Cropping benign: 100%|██████████| 437/437 [00:01<00:00, 233.96it/s]



Processing malignant: 210 images


Cropping malignant: 100%|██████████| 210/210 [00:01<00:00, 194.49it/s]


CROPPING COMPLETE
Total cropped images: 647
Class distribution:
class_name
benign       437
malignant    210
Name: count, dtype: int64

Output directory structure:
  ../../dataset/cropped_tumor_images/benign/images/     - Cropped images
  ../../dataset/cropped_tumor_images/benign/masks/      - Cropped masks
  ../../dataset/cropped_tumor_images/malignant/images/  - Cropped images
  ../../dataset/cropped_tumor_images/malignant/masks/   - Cropped masks


,image_path,mask_path,class_name,crop_image_path,crop_mask_path,bbox_xyxy
0,../../dataset/BUSI_Jpeg/benign/benign (328).png,../../dataset/BUSI_Jpeg/benign_mask/benign (32...,benign,../../dataset/cropped_tumor_images/benign/imag...,../../dataset/cropped_tumor_images/benign/mask...,"[360, 126, 443, 164]"
1,../../dataset/BUSI_Jpeg/benign/benign (282).png,../../dataset/BUSI_Jpeg/benign_mask/benign (28...,benign,../../dataset/cropped_tumor_images/benign/imag...,../../dataset/cropped_tumor_images/benign/mask...,"[170, 22, 333, 133]"
2,../../dataset/BUSI_Jpeg/benign/benign (144).png,../../dataset/BUSI_Jpeg/benign_mask/benign (14...,benign,../../dataset/cropped_tumor_images/benign/imag...,../../dataset/cropped_tumor_images/benign/mask...,"[188, 102, 235, 133]"
3,../../dataset/BUSI_Jpeg/benign/benign (82).png,../../dataset/BUSI_Jpeg/benign_mask/benign (82...,benign,../../dataset/cropped_tumor_images/benign/imag...,../../dataset/cropped_tumor_images/benign/mask...,"[340, 177, 493, 294]"
4,../../dataset/BUSI_Jpeg/benign/benign (390).png,../../dataset/BUSI_Jpeg/benign_mask/benign (39...,benign,../../dataset/cropped_tumor_images/benign/imag...,../../dataset/cropped_tumor_images/benign/mask...,"[425, 153, 584, 227]"
